<a href="https://colab.research.google.com/github/sami01979/ai_text_detection/blob/main/ai_textdetection_deberta_v3_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q transformers datasets accelerate sentencepiece tiktoken protobuf

from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/thesis_data
!cp -n /content/train_xl.csv /content/val.csv /content/test.csv /content/drive/MyDrive/thesis_data/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead


In [8]:
import os, numpy as np, pandas as pd, torch
from google.colab import drive
drive.mount('/content/drive')
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

assert torch.cuda.is_available(), "NO GPU - change runtime type to T4 first"

DATA = "/content/" if os.path.exists("/content/train_xl.csv") else "/content/drive/MyDrive/thesis_data/"
OUT  = "/content/drive/MyDrive/thesis_run_deberta"
os.makedirs(OUT, exist_ok=True)

MODEL, MAX_LEN = "microsoft/deberta-v3-base", 256

train    = pd.read_csv(DATA + "train_xl.csv")
val_full = pd.read_csv(DATA + "val.csv")
val_fast = val_full.sample(4000, random_state=42)
test     = pd.read_csv(DATA + "test.csv")
print("rows:", len(train), len(val_full), len(test))

tok = AutoTokenizer.from_pretrained(MODEL)

def prep(df):
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    return ds.remove_columns(["text"])

tr, va_fast, va, te = prep(train), prep(val_fast), prep(val_full), prep(test)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
model = model.float()
print("model dtype:", next(model.parameters()).dtype)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir=OUT,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=400,
    eval_strategy="steps", eval_steps=1000,
    save_strategy="steps", save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va_fast,
                  processing_class=tok, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=metrics)

has_ckpt = any(d.startswith("checkpoint") for d in os.listdir(OUT))
trainer.train(resume_from_checkpoint=True if has_ckpt else None)
trainer.save_model(OUT + "/final_model_deberta_v1")
tok.save_pretrained(OUT + "/final_model_deberta_v1")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rows: 100000 9951 9952


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9951 [00:00<?, ? examples/s]

Map:   0%|          | 0/9952 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/thesis_run_deberta/final_model_deberta_v1/tokenizer_config.json',
 '/content/drive/MyDrive/thesis_run_deberta/final_model_deberta_v1/tokenizer.json')

In [9]:
from scipy.special import softmax
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

val_out = trainer.predict(va)
val_p = softmax(val_out.predictions, axis=-1)[:, 1]
best_t, best_acc = 0.5, 0
for t in np.arange(0.05, 0.96, 0.01):
    acc = accuracy_score(val_out.label_ids, (val_p >= t).astype(int))
    if acc > best_acc: best_t, best_acc = t, acc
print("best threshold (VALIDATION):", round(best_t, 2), "| val acc:", round(best_acc, 4))

test_out = trainer.predict(te)
test_p = softmax(test_out.predictions, axis=-1)[:, 1]
y = test_out.label_ids
final_pred = (test_p >= best_t).astype(int)
print("TEST acc @0.5   :", round(accuracy_score(y, (test_p >= 0.5).astype(int)), 4))
print("TEST acc @tuned :", round(accuracy_score(y, final_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y, test_p), 4))
print(confusion_matrix(y, final_pred))
print(classification_report(y, final_pred, target_names=["Human", "AI"], digits=4))

best threshold (VALIDATION): 0.95 | val acc: 0.9597


TEST acc @0.5   : 0.947
TEST acc @tuned : 0.9613
ROC-AUC: 0.994
[[4712  264]
 [ 121 4855]]
              precision    recall  f1-score   support

       Human     0.9750    0.9469    0.9608      4976
          AI     0.9484    0.9757    0.9619      4976

    accuracy                         0.9613      9952
   macro avg     0.9617    0.9613    0.9613      9952
weighted avg     0.9617    0.9613    0.9613      9952



In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

path = "/content/drive/MyDrive/thesis_run_deberta/final_model_deberta_v1"
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

def detect(text, threshold=0.5):
    inputs = tok(text, truncation=True, max_length=256, return_tensors="pt").to(device)
    with torch.no_grad():
        p = torch.softmax(model(**inputs).logits, dim=-1)[0][1].item()
    print(("AI-generated" if p >= threshold else "Human-written"), f"(AI probability: {p:.1%})")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [11]:
best_t2, best_acc2 = 0.5, 0
for t in np.arange(0.50, 0.9991, 0.001):
    acc = accuracy_score(val_out.label_ids, (val_p >= t).astype(int))
    if acc > best_acc2: best_t2, best_acc2 = t, acc
print("best threshold (VALIDATION):", round(best_t2, 3), "| val acc:", round(best_acc2, 4))
print("TEST acc @that threshold:", round(accuracy_score(y, (test_p >= best_t2).astype(int)), 4))
print(confusion_matrix(y, (test_p >= best_t2).astype(int)))

best threshold (VALIDATION): 0.995 | val acc: 0.962
TEST acc @that threshold: 0.963
[[4810  166]
 [ 202 4774]]
